<a href="https://colab.research.google.com/github/leman-cap13/NLP_projects/blob/main/TransformerArchitecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Load Dataset

In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "Helsinki-NLP/opus_books",
    "en-es"
)



In [ ]:
print(dataset)

In [ ]:
dataset["train"][4]

In [ ]:
def extract_text(example):
    return {
        "source_text": example["translation"]["en"],
        "target_text": example["translation"]["es"]
    }

In [ ]:
dataset = dataset.map(
    extract_text,
    remove_columns=dataset["train"].column_names
)

In [ ]:
dataset

In [ ]:
dataset["train"][4]

#Tokenizer

In [ ]:
!pip install -q tokenizers

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.normalizers import Sequence, Lowercase
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

In [ ]:
special_tokens = [
    "<pad>",
    "<unk>",
    "<sos>",
    "<eos>"
]

In [ ]:
tokenizer = Tokenizer(
    BPE(unk_token="<unk>")
)

In [ ]:
tokenizer.pre_tokenizer = ByteLevel(
    add_prefix_space=False
)

In [ ]:
trainer = BpeTrainer(
    vocab_size=16_000,
    min_frequency=2,
    special_tokens=special_tokens,
    initial_alphabet=ByteLevel.alphabet(),
    show_progress=True
)

In [ ]:
def get_training_corpus(batch_size=1000):
    train_data = dataset["train"]

    for start_index in range(0, len(train_data), batch_size):
        batch = train_data[start_index:start_index + batch_size]

        english_texts = batch["source_text"]
        spanish_texts = batch["target_text"]

        yield english_texts + spanish_texts

In [ ]:
tokenizer.train_from_iterator(
    get_training_corpus(),
    trainer=trainer,
    length=len(dataset["train"]) * 2
)

In [ ]:
tokenizer.get_vocab_size()

In [ ]:
example_text = "I like playing soccer."

encoded = tokenizer.encode(example_text)

In [ ]:
encoded.tokens

In [ ]:
encoded.ids

In [ ]:
tokenizer.decoder = ByteLevelDecoder()

In [ ]:
decoded = tokenizer.decode(encoded.ids)

decoded

#Encoder Decoder preprocessing

In [ ]:
PAD_ID = tokenizer.token_to_id("<pad>")
UNK_ID = tokenizer.token_to_id("<unk>")
SOS_ID = tokenizer.token_to_id("<sos>")
EOS_ID = tokenizer.token_to_id("<eos>")

In [ ]:
EOS_ID

In [ ]:
#target text => sos + text + eos
#

In [ ]:
0, *[1,2,3,4], 4

In [ ]:
def encode_row(row):
    source_ids = (
        tokenizer.encode(row["source_text"]).ids + [EOS_ID] )

    full_target_ids = [
        SOS_ID,
        *tokenizer.encode(row["target_text"]).ids,
        EOS_ID ]

    decoder_input_ids = full_target_ids[:-1]

    label_ids = full_target_ids[1:]

    return {
        "source_ids": source_ids,
        "decoder_input_ids": decoder_input_ids,
        "label_ids": label_ids
    }

In [ ]:
        # source_text
        # target_text
        # +
        # "source_ids": source_ids,
        # "decoder_input_ids": decoder_input_ids,
        # "label_ids": label_ids

In [ ]:
dataset["train"].column_names

In [ ]:
tokenized_dataset = dataset.map(
    encode_row,
    remove_columns=dataset["train"].column_names
)

In [ ]:
tokenized_dataset

In [ ]:
tokenized_dataset["train"][4]

In [ ]:
split_dataset = tokenized_dataset["train"].train_test_split(
    test_size=0.2,
    seed=42
)

In [ ]:
validation_test_split = split_dataset["test"].train_test_split(
    test_size=0.5,
    seed=42
)

In [ ]:
from datasets import DatasetDict

tokenized_dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": validation_test_split["train"],
    "test": validation_test_split["test"]
})

In [ ]:
print(tokenized_dataset)

In [ ]:
# row = tokenized_dataset["train"][4]

# print("Source tokens:")
# print([
#     tokenizer.id_to_token(token_id)
#     for token_id in row["source_ids"]
# ])

# print("\nDecoder input tokens:")
# print([
#     tokenizer.id_to_token(token_id)
#     for token_id in row["decoder_input_ids"]
# ])

# print("\nLabel tokens:")
# print([
#     tokenizer.id_to_token(token_id)
#     for token_id in row["label_ids"]
# ])

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

In [ ]:
MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128


def collate_fn(rows):
    source_sequences = []
    decoder_input_sequences = []
    label_sequences = []

    for row in rows:
        source_ids = list(
            row["source_ids"][:MAX_SOURCE_LENGTH]
        )


        if source_ids and source_ids[-1] != EOS_ID:
            source_ids[-1] = EOS_ID


        decoder_input_ids = list(row["decoder_input_ids"][:MAX_TARGET_LENGTH] )

        label_ids = list( row["label_ids"][:MAX_TARGET_LENGTH])


        if (len(row["label_ids"]) > MAX_TARGET_LENGTH and label_ids ):
            label_ids[-1] = EOS_ID

        source_sequences.append(
            torch.tensor(
                source_ids,
                dtype=torch.long
            )
        )

        decoder_input_sequences.append(
            torch.tensor(
                decoder_input_ids,
                dtype=torch.long
            )
        )

        label_sequences.append(
            torch.tensor(
                label_ids,
                dtype=torch.long
            )
        )

    source_ids = pad_sequence(
        source_sequences,
        batch_first=True,
        padding_value=PAD_ID
    )

    decoder_input_ids = pad_sequence(
        decoder_input_sequences,
        batch_first=True,
        padding_value=PAD_ID
    )

    labels = pad_sequence(
        label_sequences,
        batch_first=True,
        padding_value=-100
    )

    return {
        "source_ids": source_ids,
        "decoder_input_ids": decoder_input_ids,
        "labels": labels
    }

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss(
    ignore_index=-100
)

In [ ]:
BATCH_SIZE = 4

train_loader = DataLoader(
    tokenized_dataset["train"],
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

validation_loader = DataLoader(
    tokenized_dataset["validation"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    tokenized_dataset["test"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
batch = next(iter(train_loader))

In [ ]:
batch

In [ ]:
# print(
#     "Source shape:",
#     batch["source_ids"].shape
# )

# print(
#     "Decoder input shape:",
#     batch["decoder_input_ids"].shape
# )

# print(
#     "Labels shape:",
#     batch["labels"].shape
# )

#Padding mask vs causal mask

In [ ]:
def create_padding_mask(token_ids, pad_id):
    return token_ids == pad_id

In [ ]:
source_padding_mask = create_padding_mask(
    batch["source_ids"],
    PAD_ID
)

target_padding_mask = create_padding_mask(
    batch["decoder_input_ids"],
    PAD_ID
)

In [ ]:
source_padding_mask

In [ ]:
target_padding_mask

In [ ]:
# [ F, T, T,T]
#.[ F, F, T, T]
# [ F, F, F, T]

In [ ]:
def create_causal_mask(sequence_length):
    return torch.triu(
        torch.ones(
            sequence_length,
            sequence_length,
            dtype=torch.bool
        ),
        diagonal=1
    )

In [ ]:
batch["decoder_input_ids"].size()

In [ ]:
target_length = batch["decoder_input_ids"].size(1)

In [ ]:
target_length

In [ ]:
causal_mask = create_causal_mask(target_length)

In [ ]:
causal_mask

In [ ]:
causal_mask.shape

In [ ]:
batch = next(iter(train_loader))

source_ids = batch["source_ids"]
decoder_input_ids = batch["decoder_input_ids"]
labels = batch["labels"]


source_padding_mask = create_padding_mask(  #encoder self attention, cross attention
    source_ids,
    PAD_ID
)

target_padding_mask = create_padding_mask( # decoder self attention
    decoder_input_ids,
    PAD_ID
)

causal_mask = create_causal_mask( # decoder self attention
    decoder_input_ids.size(1)
)

In [ ]:
# Encoder self-attention
#     → source_padding_mask

# Decoder self-attention
#     → causal_mask
#     → target_padding_mask

# Decoder cross-attention
#     → source_padding_mask

In [ ]:
print("Source IDs:", source_ids.shape)
print("Decoder input IDs:", decoder_input_ids.shape)

print(
    "Source padding mask:",
    source_padding_mask.shape
)

print(
    "Target padding mask:",
    target_padding_mask.shape
)

print(
    "Causal mask:",
    causal_mask.shape
)

#Token Embedding

In [ ]:
import math
import torch
import torch.nn as nn

In [ ]:
class TokenEmbedding(nn.Module):
    def __init__( self, vocab_size, embed_dim,  pad_id):
        super().__init__()

        self.embed_dim = embed_dim

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id
        )

    def forward(self, token_ids):
        embeddings = self.embedding(token_ids)

        embeddings = embeddings * math.sqrt(self.embed_dim) # [0.023, 0.07, 0.003] + [0.7, 0.78, 0,98]

        return embeddings

In [ ]:
VOCAB_SIZE = tokenizer.get_vocab_size()
VOCAB_SIZE

In [ ]:
EMBED_DIM = 512

In [ ]:
token_embedding = TokenEmbedding(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    pad_id=PAD_ID
)

In [ ]:
source_ids = batch["source_ids"]
decoder_input_ids = batch["decoder_input_ids"]

In [ ]:
source_embeddings = token_embedding(source_ids)

decoder_embeddings = token_embedding(
    decoder_input_ids
)

In [ ]:
source_embeddings

In [ ]:
print("Source IDs shape:")
print(source_ids.shape)

print("\nSource embeddings shape:")
print(source_embeddings.shape)

print("\nDecoder IDs shape:")
print(decoder_input_ids.shape)

print("\nDecoder embeddings shape:")
print(decoder_embeddings.shape)

#Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_length=512, dropout=0.1):
        super().__init__()

        self.dropout = nn.Dropout(dropout)

        positions = torch.arange( max_length, dtype=torch.float ).unsqueeze(1)

        dimension_indices = torch.arange( 0,  embed_dim,  2,  dtype=torch.float )

        denominator = 10000 ** (dimension_indices / embed_dim)

        positional_encoding = torch.zeros(
            max_length,
            embed_dim
        )


        positional_encoding[:, 0::2] = torch.sin(
            positions / denominator
        )


        positional_encoding[:, 1::2] = torch.cos(
            positions / denominator
        )


        positional_encoding = positional_encoding.unsqueeze(0)

        self.register_buffer(
            "positional_encoding",
            positional_encoding
        )

    def forward(self, embeddings):
        sequence_length = embeddings.size(1)

        embeddings = embeddings + self.positional_encoding[:, :sequence_length ]

        return self.dropout(embeddings)

In [ ]:
positional_encoding = PositionalEncoding(
    embed_dim=EMBED_DIM,
    max_length=512,
    dropout=0.1
)

In [ ]:
source_embeddings_with_position = positional_encoding(
    source_embeddings
)

decoder_embeddings_with_position = positional_encoding(
    decoder_embeddings
)

In [ ]:
print(
    "Source embedding shape:",
    source_embeddings.shape
)

print(
    "Source + position shape:",
    source_embeddings_with_position.shape
)

print(
    "Decoder embedding shape:",
    decoder_embeddings.shape
)

print(
    "Decoder + position shape:",
    decoder_embeddings_with_position.shape
)

#Scaled Dot-Product Attention

In [ ]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()

        self.dropout = nn.Dropout(dropout)

    def forward(self, query,  key, value, attention_mask=None):


        key_dim = query.size(-1)

        scores = (query @ key.transpose(-2, -1)) / math.sqrt(key_dim)


        if attention_mask is not None:
            scores = scores.masked_fill(
                attention_mask,
                float("-inf")
            )

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        attention_weights = self.dropout(
            attention_weights
        )

        context = attention_weights @ value


        return context, attention_weights

In [ ]:
attention = ScaledDotProductAttention(
    dropout=0.1
)

In [ ]:
context, attention_weights = attention(
    query=source_embeddings_with_position,
    key=source_embeddings_with_position,
    value=source_embeddings_with_position
)

In [ ]:
context

In [ ]:
attention_weights

In [ ]:
print(
    "Query shape:",
    source_embeddings_with_position.shape
)

print(
    "Context shape:",
    context.shape
)

print(
    "Attention weights shape:",
    attention_weights.shape
)

In [ ]:
source_attention_mask = (
    source_padding_mask.unsqueeze(1)
)

In [ ]:
context, attention_weights = attention(
    query=source_embeddings_with_position,
    key=source_embeddings_with_position,
    value=source_embeddings_with_position,
    attention_mask=source_attention_mask
)

In [ ]:
print("Source IDs:", source_ids.shape)

print(
    "Embeddings:",
    source_embeddings_with_position.shape
)

print(
    "Mask:",
    source_attention_mask.shape
)

print("Context:", context.shape)

print(
    "Attention weights:",
    attention_weights.shape
)

#Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear( embed_dim, embed_dim )

        self.k_proj = nn.Linear( embed_dim, embed_dim )

        self.v_proj = nn.Linear( embed_dim, embed_dim )

        self.attention = ScaledDotProductAttention(dropout=dropout)

        self.out_proj = nn.Linear(embed_dim, embed_dim )

    def split_heads(self, x):
        batch_size, sequence_length, _ = x.shape #[batch_size, seq_len, embed_dim]=> x.shape

        x = x.reshape(
            batch_size,
            sequence_length,
            self.num_heads,
            self.head_dim
        )

        x = x.transpose(1, 2) # [b, seq_len, num_head, head_dim] => [b, num_head, seq_len, head_dim]

        return x

    def combine_heads(self, x):
        batch_size, _, sequence_length, _ = x.shape

        x = x.transpose(1, 2) # [b,s,n_h, h_dim]

        x = x.reshape(
            batch_size,
            sequence_length,
            self.embed_dim
        )

        return x

    def forward( self, query,  key, value, attention_mask=None,  key_padding_mask=None ):

        query = self.q_proj(query)
        key = self.k_proj(key)
        value = self.v_proj(value)


        query = self.split_heads(query)
        key = self.split_heads(key)
        value = self.split_heads(value)

        combined_mask = None

        if attention_mask is not None:
            if attention_mask.dim() == 2:
                combined_mask = attention_mask[ None, None, :, : ] #[1,1, S,S]
            elif attention_mask.dim() == 3:
                combined_mask = attention_mask.unsqueeze(1) # [ B, 1, S]
            else:
                combined_mask = attention_mask


        if key_padding_mask is not None:
            padding_mask = key_padding_mask[ :, None, None, : ] #[B, 1, 1, S]

            if combined_mask is None:
                combined_mask = padding_mask
            else:
                combined_mask = (
                    combined_mask | padding_mask # for decoder self attention causal OR padding mask
                )


        context, attention_weights = self.attention(
            query=query,
            key=key,
            value=value,
            attention_mask=combined_mask
        )


        context = self.combine_heads(context)

        output = self.out_proj(context)

        return output, attention_weights

In [ ]:
NUM_HEADS = 8

multi_head_attention = MultiHeadAttention(
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    dropout=0.1
)

In [ ]:
mha_output, mha_weights = multi_head_attention(
    query=source_embeddings_with_position,
    key=source_embeddings_with_position,
    value=source_embeddings_with_position,
    key_padding_mask=source_padding_mask
)

In [ ]:
mha_weights.shape

In [ ]:
print(
    "Input:",
    source_embeddings_with_position.shape
)

print(
    "MHA output:",
    mha_output.shape
)

print(
    "MHA weights:",
    mha_weights.shape
)

#Feed-Forward Network

In [ ]:
class FeedForward(nn.Module):
    def __init__( self, embed_dim,  hidden_dim,  dropout=0.1): #512 2048 512
        super().__init__()

        self.linear1 = nn.Linear(
            embed_dim,
            hidden_dim
        )

        self.activation = nn.ReLU()

        self.dropout = nn.Dropout(dropout)

        self.linear2 = nn.Linear(
            hidden_dim,
            embed_dim
        )

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)

        return x

In [ ]:
FF_HIDDEN_DIM = 2048

feed_forward = FeedForward(
    embed_dim=EMBED_DIM,
    hidden_dim=FF_HIDDEN_DIM,
    dropout=0.1
)

In [ ]:
ff_output = feed_forward(mha_output)

In [ ]:
print("MHA output shape:")
print(mha_output.shape)

print("\nFeed-forward output shape:")
print(ff_output.shape)

#Encoder Block

## Block

In [ ]:
class EncoderBlock(nn.Module):
    def __init__( self, embed_dim,  num_heads, ff_hidden_dim,  dropout=0.1 ):
        super().__init__()

        self.self_attention = MultiHeadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout
        )

        self.feed_forward = FeedForward(
            embed_dim=embed_dim,
            hidden_dim=ff_hidden_dim,
            dropout=dropout
        )

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward( self,  x, source_padding_mask=None ):

        attention_output, attention_weights = self.self_attention(
            query=x,
            key=x,
            value=x,
            key_padding_mask=source_padding_mask
        )


        x = self.norm1( x + self.dropout1(attention_output) )

        feed_forward_output = self.feed_forward(x)


        x = self.norm2( x + self.dropout2(feed_forward_output) )

        return x, attention_weights

In [ ]:
encoder_block = EncoderBlock(
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_hidden_dim=FF_HIDDEN_DIM,
    dropout=0.1
)

In [ ]:
encoder_input = positional_encoding(
    source_embeddings
)

In [ ]:
encoder_output, encoder_attention_weights = encoder_block(
    x=encoder_input,
    source_padding_mask=source_padding_mask
)

In [ ]:
print("Encoder input shape:")
print(encoder_input.shape)

print("\nEncoder output shape:")
print(encoder_output.shape)

print("\nAttention weights shape:")
print(encoder_attention_weights.shape)

##Encoder

In [ ]:
class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads,
        ff_hidden_dim,
        num_layers,
        max_length,
        pad_id,
        dropout=0.1
    ):

        super().__init__()

        self.token_embedding = TokenEmbedding(
            vocab_size=vocab_size,
            embed_dim=embed_dim,
            pad_id=pad_id
        )

        self.positional_encoding = PositionalEncoding(
            embed_dim=embed_dim,
            max_length=max_length,
            dropout=dropout
        )

        self.layers = nn.ModuleList([
            EncoderBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                ff_hidden_dim=ff_hidden_dim,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

    def forward( self, source_ids, source_padding_mask=None ):

        x = self.token_embedding(source_ids)

        # Token embedding + positional encoding
        x = self.positional_encoding(x)

        all_attention_weights = []


        for layer in self.layers:
            x, attention_weights = layer( x=x,  source_padding_mask=source_padding_mask )


            all_attention_weights.append( attention_weights )

        return x, all_attention_weights

In [ ]:
NUM_ENCODER_LAYERS = 6
MAX_LENGTH = 512

In [ ]:
encoder = Encoder(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_hidden_dim=FF_HIDDEN_DIM,
    num_layers=NUM_ENCODER_LAYERS,
    max_length=MAX_LENGTH,
    pad_id=PAD_ID,
    dropout=0.1
)

In [ ]:
encoder_output, encoder_attention_weights = encoder(
    source_ids=source_ids,
    source_padding_mask=source_padding_mask
)

In [ ]:
print("Source IDs shape:")
print(source_ids.shape)

print("\nEncoder output shape:")
print(encoder_output.shape)

print("\nNumber of encoder layers:")
print(len(encoder_attention_weights))

print("\nFirst layer attention shape:")
print(encoder_attention_weights[0].shape)

#Decoder Block

##Block

In [ ]:
class DecoderBlock(nn.Module):
    def __init__( self,  embed_dim, num_heads, ff_hidden_dim,  dropout=0.1 ):

        super().__init__()


        self.masked_self_attention = MultiHeadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout
        )


        self.cross_attention = MultiHeadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout
        )

        self.feed_forward = FeedForward(
            embed_dim=embed_dim,
            hidden_dim=ff_hidden_dim,
            dropout=dropout
        )

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.norm3 = nn.LayerNorm(embed_dim)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(
        self,
        x,
        encoder_output,
        causal_mask=None,
        target_padding_mask=None,
        source_padding_mask=None
    ):

        self_attention_output, self_attention_weights = (
            self.masked_self_attention(
                query=x,
                key=x,
                value=x,
                attention_mask=causal_mask,
                key_padding_mask=target_padding_mask
            )
        )


        x = self.norm1( x + self.dropout1(self_attention_output) )


        cross_attention_output, cross_attention_weights = (
            self.cross_attention(
                query=x,
                key=encoder_output,
                value=encoder_output,
                key_padding_mask=source_padding_mask
            )
        )


        x = self.norm2( x + self.dropout2(cross_attention_output)  )


        feed_forward_output = self.feed_forward(x)


        x = self.norm3( x + self.dropout3(feed_forward_output) )

        return (
            x,
            self_attention_weights,
            cross_attention_weights
        )

In [ ]:
decoder_block = DecoderBlock(
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_hidden_dim=FF_HIDDEN_DIM,
    dropout=0.1
)

In [ ]:
target_embeddings = token_embedding(
    decoder_input_ids
)

In [ ]:
decoder_input = positional_encoding(
    target_embeddings
)

In [ ]:
(
    decoder_output,
    decoder_self_attention_weights,
    cross_attention_weights

) = decoder_block(
    x=decoder_input,
    encoder_output=encoder_output,
    causal_mask=causal_mask,
    target_padding_mask=target_padding_mask,
    source_padding_mask=source_padding_mask
)


In [ ]:
print("Decoder input:", decoder_input.shape)
print("Encoder output:", encoder_output.shape)
print("Decoder output:", decoder_output.shape)

print(
    "Self-attention:",
    decoder_self_attention_weights.shape
)

print(
    "Cross-attention:",
    cross_attention_weights.shape
)

##Decoder

In [ ]:
class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads,
        ff_hidden_dim,
        num_layers,
        max_length,
        pad_id,
        dropout=0.1
    ):
        super().__init__()

        self.token_embedding = TokenEmbedding(
            vocab_size=vocab_size,
            embed_dim=embed_dim,
            pad_id=pad_id
        )

        self.positional_encoding = PositionalEncoding(
            embed_dim=embed_dim,
            max_length=max_length,
            dropout=dropout
        )

        self.layers = nn.ModuleList([
            DecoderBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                ff_hidden_dim=ff_hidden_dim,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

    def forward(
        self,
        decoder_input_ids,
        encoder_output,
        causal_mask=None,
        target_padding_mask=None,
        source_padding_mask=None
    ):

        x = self.token_embedding(decoder_input_ids )

        x = self.positional_encoding(x)

        all_self_attention_weights = []
        all_cross_attention_weights = []

        for layer in self.layers:
            (x, self_attention_weights, cross_attention_weights ) = layer(
                x=x,
                encoder_output=encoder_output,
                causal_mask=causal_mask,
                target_padding_mask=target_padding_mask,
                source_padding_mask=source_padding_mask
            )


            all_self_attention_weights.append( self_attention_weights )

            all_cross_attention_weights.append( cross_attention_weights )

        return (
            x,
            all_self_attention_weights,
            all_cross_attention_weights
        )

In [ ]:
NUM_DECODER_LAYERS = 6

In [ ]:
decoder = Decoder(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_hidden_dim=FF_HIDDEN_DIM,
    num_layers=NUM_DECODER_LAYERS,
    max_length=MAX_LENGTH,
    pad_id=PAD_ID,
    dropout=0.1
)

In [ ]:
(
    decoder_output,
    decoder_self_attention_weights,
    decoder_cross_attention_weights

) = decoder(
    decoder_input_ids=decoder_input_ids,
    encoder_output=encoder_output,
    causal_mask=causal_mask,
    target_padding_mask=target_padding_mask,
    source_padding_mask=source_padding_mask
)

In [ ]:
decoder_self_attention_weights[0].shape

In [ ]:
print("Source IDs:")
print(source_ids.shape)

print("\nDecoder input IDs:")
print(decoder_input_ids.shape)

print("\nEncoder output:")
print(encoder_output.shape)

print("\nDecoder output:")
print(decoder_output.shape)

print("\nNumber of decoder layers:")
print(len(decoder_self_attention_weights))

print("\nFirst decoder self-attention shape:")
print(decoder_self_attention_weights[0].shape)

print("\nFirst decoder cross-attention shape:")
print(decoder_cross_attention_weights[0].shape)

#Transformer

In [ ]:
class Transformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads,
        ff_hidden_dim,
        num_encoder_layers,
        num_decoder_layers,
        max_length,
        pad_id,
        dropout=0.1
    ):
        super().__init__()

        self.pad_id = pad_id

        self.encoder = Encoder(
            vocab_size=vocab_size,
            embed_dim=embed_dim,
            num_heads=num_heads,
            ff_hidden_dim=ff_hidden_dim,
            num_layers=num_encoder_layers,
            max_length=max_length,
            pad_id=pad_id,
            dropout=dropout
        )

        self.decoder = Decoder(
            vocab_size=vocab_size,
            embed_dim=embed_dim,
            num_heads=num_heads,
            ff_hidden_dim=ff_hidden_dim,
            num_layers=num_decoder_layers,
            max_length=max_length,
            pad_id=pad_id,
            dropout=dropout
        )

        self.output_layer = nn.Linear(
            in_features=embed_dim,
            out_features=vocab_size
        )

    def forward( self, source_ids, decoder_input_ids ):

        source_padding_mask = create_padding_mask( source_ids, self.pad_id )  # [B, S]

        target_padding_mask = create_padding_mask( decoder_input_ids,  self.pad_id )  # [B, S]


        causal_mask = create_causal_mask(decoder_input_ids.size(1)).to(decoder_input_ids.device)   # [T, T]

        # Encoder
        (encoder_output, encoder_attention_weights ) = self.encoder(            # [B, S] → [B, S, E]
            source_ids=source_ids,
            source_padding_mask=source_padding_mask
        )


        # Decoder
        (decoder_output, decoder_self_attention_weights, decoder_cross_attention_weights ) = self.decoder(     # [B, T] → [B, T, E]
            decoder_input_ids=decoder_input_ids,
            encoder_output=encoder_output,
            causal_mask=causal_mask,
            target_padding_mask=target_padding_mask,
            source_padding_mask=source_padding_mask
        )


        logits = self.output_layer( decoder_output )

        return (
            logits,
            encoder_attention_weights,
            decoder_self_attention_weights,
            decoder_cross_attention_weights
        )

In [ ]:
model = Transformer(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_hidden_dim=FF_HIDDEN_DIM,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    max_length=MAX_LENGTH,
    pad_id=PAD_ID,
    dropout=0.1
)

In [ ]:
(
    logits,
    encoder_attention_weights,
    decoder_self_attention_weights,
    decoder_cross_attention_weights
) = model(
    source_ids=source_ids,
    decoder_input_ids=decoder_input_ids
)

In [ ]:
print("Source IDs:")
print(source_ids.shape)

print("\nDecoder input IDs:")
print(decoder_input_ids.shape)

print("\nLabels:")
print(labels.shape)

print("\nLogits:")
print(logits.shape)

#Train

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

model = model.to(device)

In [ ]:
loss_fn = nn.CrossEntropyLoss(
    ignore_index=-100,
    label_smoothing=0.1
)

In [ ]:
loss = loss_fn(
    logits.reshape(-1, VOCAB_SIZE),  #[B, T, V] => [B*T, V]
    labels.reshape(-1) #[B, T] => [B*T]
)

In [ ]:
print("Loss:", loss.item()) #math.log(vocab_size)

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0,
    betas=(0.9, 0.98),
    eps=1e-9
)

In [ ]:
def get_noam_lr(step, embed_dim, warmup_steps, factor=0.2):
    step = max(step, 1)

    lr = factor * (embed_dim ** -0.5) * min(
        step ** -0.5,
        step * (warmup_steps ** -1.5)
    )

    return lr

In [ ]:
from tqdm.auto import tqdm

def train_one_epoch(
    model,
    data_loader,
    optimizer,
    loss_fn,
    device,
    vocab_size,
    embed_dim,
    warmup_steps,
    global_step
):
    model.train()

    total_loss = 0.0
    total_valid_tokens = 0

    progress_bar = tqdm(
        data_loader,
        desc="Training",
        leave=False
    )

    for batch in progress_bar:

        source_ids = batch["source_ids"].to(device)
        decoder_input_ids = batch["decoder_input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        logits, _, _, _ = model(
            source_ids=source_ids,
            decoder_input_ids=decoder_input_ids
        )

        #input shape:  [N, vocab_size]      target shape: [N]
        loss = loss_fn(
            logits.reshape(-1, vocab_size),  #[B, T, V] => [B*T, V]
            labels.reshape(-1) # [B, T] => [B*T]
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        global_step += 1

        current_lr = get_noam_lr(
            step=global_step,
            embed_dim=embed_dim,
            warmup_steps=warmup_steps,
            factor=0.2
            )

        for param_group in optimizer.param_groups:
            param_group["lr"] = current_lr

        optimizer.step()

        valid_tokens = (
            labels != -100
        ).sum().item()

        total_loss += loss.item() * valid_tokens
        total_valid_tokens += valid_tokens

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{current_lr:.6f}"
        )

    average_loss = total_loss / total_valid_tokens

    return average_loss, global_step


def evaluate(
    model,
    data_loader,
    loss_fn,
    device,
    vocab_size
):
    model.eval()

    total_loss = 0.0
    total_valid_tokens = 0

    with torch.no_grad():

        for batch in data_loader:

            source_ids = batch[ "source_ids" ].to(device)

            decoder_input_ids = batch["decoder_input_ids" ].to(device)

            labels = batch[ "labels" ].to(device)


            logits, _, _, _ = model(
                source_ids=source_ids,
                decoder_input_ids=decoder_input_ids
            )


            loss = loss_fn(
                logits.reshape(-1, vocab_size),
                labels.reshape(-1)
            )


            valid_tokens = (
                labels != -100
            ).sum().item()

            total_loss += loss.item() * valid_tokens
            total_valid_tokens += valid_tokens


    average_loss = (
        total_loss / total_valid_tokens
    )

    return average_loss



def train_model(
    model,
    train_loader,
    validation_loader,
    optimizer,
    loss_fn,
    device,
    vocab_size,
    embed_dim,
    warmup_steps,
    num_epochs
):
    train_losses = []
    validation_losses = []

    best_validation_loss = float("inf")
    global_step = 0

    for epoch in range(1, num_epochs + 1):

        train_loss, global_step = train_one_epoch(
            model=model,
            data_loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=device,
            vocab_size=vocab_size,
            embed_dim=embed_dim,
            warmup_steps=warmup_steps,
            global_step=global_step
        )

        validation_loss = evaluate(
            model=model,
            data_loader=validation_loader,
            loss_fn=loss_fn,
            device=device,
            vocab_size=vocab_size
        )

        train_losses.append(train_loss)
        validation_losses.append(validation_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch:02d}/{num_epochs} | "
            f"Step: {global_step} | "
            f"LR: {current_lr:.6f} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Validation Loss: {validation_loss:.4f}"
        )

        if validation_loss < best_validation_loss:

            best_validation_loss = validation_loss

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "global_step": global_step,
                    "validation_loss": validation_loss,
                },
                "best_transformer_model.pt"
            )

            print("Best model saved.")

    return train_losses, validation_losses

In [ ]:
NUM_EPOCHS = 3

TOTAL_STEPS = NUM_EPOCHS * len(train_loader)

WARMUP_STEPS = min(
    4000,
    max(100, int(0.1 * TOTAL_STEPS))
)



In [ ]:
train_losses, validation_losses = train_model(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    warmup_steps=WARMUP_STEPS,
    num_epochs=NUM_EPOCHS
)

#Translate

In [ ]:
model.eval()

In [ ]:
def translate(
    model,
    text,
    tokenizer,
    device,
    sos_id,
    eos_id,
    max_source_length=128,
    max_target_length=128
):
    model.eval()

    source_token_ids = tokenizer.encode(text).ids

    source_token_ids = source_token_ids[ :max_source_length - 1]

    source_token_ids = source_token_ids + [eos_id]

    source_ids = torch.tensor(
        [source_token_ids],
        dtype=torch.long,
        device=device
    )


    generated_ids = torch.tensor(
        [[sos_id]],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():

        for _ in range(max_target_length):

            logits, _, _, _ = model(
                source_ids=source_ids,
                decoder_input_ids=generated_ids
            )


            next_token_logits = logits[:, -1, :]


            next_token_id = next_token_logits.argmax(
                dim=-1,
                keepdim=True
            )


            generated_ids = torch.cat(
                [generated_ids, next_token_id],
                dim=1
            )


            if next_token_id.item() == eos_id:
                break

    predicted_ids = generated_ids[0].tolist()

    translated_text = tokenizer.decode(
        predicted_ids,
        skip_special_tokens=True
    )

    return translated_text

In [ ]:
sentences = [
    "I love my family.",
    "The boy is reading a book.",
    "She lives in a small house.",
    "We went to the city yesterday."
]

for sentence in sentences:

    translation = translate(
        model=model,
        text=sentence,
        tokenizer=tokenizer,
        device=device,
        sos_id=SOS_ID,
        eos_id=EOS_ID
    )

    print("English:", sentence)
    print("Spanish:", translation)
    print()

#One-batch sanity check

In [ ]:
DEBUG_EMBED_DIM = 128
DEBUG_NUM_HEADS = 4
DEBUG_FF_HIDDEN_DIM = 512
DEBUG_NUM_ENCODER_LAYERS = 2
DEBUG_NUM_DECODER_LAYERS = 2
DEBUG_MAX_LENGTH = 128

In [ ]:
debug_model = Transformer(
    vocab_size=VOCAB_SIZE,
    embed_dim=DEBUG_EMBED_DIM,
    num_heads=DEBUG_NUM_HEADS,
    ff_hidden_dim=DEBUG_FF_HIDDEN_DIM,
    num_encoder_layers=DEBUG_NUM_ENCODER_LAYERS,
    num_decoder_layers=DEBUG_NUM_DECODER_LAYERS,
    max_length=DEBUG_MAX_LENGTH,
    pad_id=PAD_ID,
    dropout=0.0
).to(device)

In [ ]:
debug_optimizer = torch.optim.Adam(
    debug_model.parameters(),
    lr=1e-3
)

In [ ]:
debug_loss_fn = nn.CrossEntropyLoss(
    ignore_index=-100
)

In [ ]:
one_batch = next(iter(train_loader))

In [ ]:
source_ids = one_batch["source_ids"].to(device)

decoder_input_ids = one_batch[
    "decoder_input_ids"
].to(device)

labels = one_batch["labels"].to(device)

In [ ]:
print("Source:", source_ids.shape)
print("Decoder input:", decoder_input_ids.shape)
print("Labels:", labels.shape)

In [ ]:
def calculate_token_accuracy(logits, labels):
    predictions = logits.argmax(dim=-1)

    valid_positions = labels != -100

    correct_predictions = (
        predictions[valid_positions]
        == labels[valid_positions]
    ).sum()

    total_predictions = valid_positions.sum()

    accuracy = (
        correct_predictions.float()
        / total_predictions.float()
    )

    return accuracy.item()

In [ ]:
NUM_STEPS = 300

debug_model.train()

for step in range(1, NUM_STEPS + 1):

    debug_optimizer.zero_grad(set_to_none=True)

    logits, _, _, _ = debug_model(
        source_ids=source_ids,
        decoder_input_ids=decoder_input_ids
    )

    loss = debug_loss_fn(
        logits.reshape(-1, VOCAB_SIZE),
        labels.reshape(-1)
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        debug_model.parameters(),
        max_norm=1.0
    )

    debug_optimizer.step()

    if step == 1 or step % 20 == 0:
        accuracy = calculate_token_accuracy(
            logits,
            labels
        )

        print(
            f"Step {step:03d} | "
            f"Loss: {loss.item():.4f} | "
            f"Token Accuracy: {accuracy:.4f}"
        )

In [ ]:
source_ids_one = source_ids[0]

source_token_ids = [
    token_id
    for token_id in source_ids_one.tolist()
    if token_id != PAD_ID
]

source_text_one = tokenizer.decode(
    source_token_ids,
    skip_special_tokens=True
)

print("Source:", source_text_one)

print(
    "Translation:",
    translate(
        model=debug_model,
        text=source_text_one,
        tokenizer=tokenizer,
        device=device,
        sos_id=SOS_ID,
        eos_id=EOS_ID,
        max_target_length=128
    )
)

In [ ]:
gold_target_ids = [
    token_id
    for token_id in labels[0].tolist()
    if token_id != -100
]

gold_target_text = tokenizer.decode(
    gold_target_ids,
    skip_special_tokens=True
)

print("Source:")
print(source_text_one)

print("\nGold translation:")
print(gold_target_text)

print("\nModel translation:")
print(
    translate(
        model=debug_model,
        text=source_text_one,
        tokenizer=tokenizer,
        device=device,
        sos_id=SOS_ID,
        eos_id=EOS_ID,
        max_target_length=128
    )
)